# Embeddings and Tokenization from Scratch

This notebook follows the core ideas from Chapter 2 of *Build a Large Language Model (From Scratch)* by Sebastian Raschka.

The goal is to understand how raw text is transformed into numerical representations (tokens and embeddings), and why this process is fundamental for Large Language Models and agentic AI systems.

In [1]:
import torch
import tiktoken

## Tokenization: Turning Text into Numbers

Large Language Models cannot process raw text directly. Neural networks are numerical, so the first step is to represent text as numbers, which are called *tokens*.

Contemporary LLMs employ subword tokenization methods (like Byte Pair Encoding) to represent language efficiently. This enables LLMs to process rare words, new vocabulary, and spelling errors without the need for an unreasonably large vocabulary.

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

len(text)

20479

In [4]:
tokens = tokenizer.encode(text)
len(tokens)

5145

## Context Windows and max_length

LLMs learn to predict the next token based on a fixed-size context window. The parameter max_length determines the number of tokens the model can see simultaneously.

This is a critical design decision:
- A small context window might overlook significant dependencies in language.
- A larger context window is more informative but more computationally expensive.

In [5]:
def create_dataset(tokens, max_length, stride):
    input_ids = []
    target_ids = []

    for i in range(0, len(tokens) - max_length, stride):
        input_ids.append(tokens[i:i + max_length])
        target_ids.append(tokens[i + 1:i + max_length + 1])

    return torch.tensor(input_ids), torch.tensor(target_ids)

In [6]:
max_length = 32
stride = 16

X, y = create_dataset(tokens, max_length, stride)

X.shape, y.shape

(torch.Size([320, 32]), torch.Size([320, 32]))

## Overlapping Windows and Stride

The `stride` argument determines how much the context window advances with each iteration.

If `stride` is less than `max_length`, the windows will overlap.

The overlap is beneficial because it enables the model to learn the same tokens in different contexts, which enhances the efficiency of learning and generalization.

The overlap is particularly beneficial when the amount of training data is limited because it enables the model to learn more samples without increasing the amount of text.

## Why Do Embeddings Encode Meaning?

Embeddings capture meaning because they are learned vectors that are optimized using neural network training.

From the perspective of a neural network, embeddings are just weight matrices that project token IDs to dense vectors. During training, tokens that appear in the same context receive the same gradient, which causes their vectors to be similar in embedding space.

As a consequence, embeddings capture semantic relationships because tokens that have similar meanings are similar in geometric space. This is why embeddings are so effective in downstream applications such as search, clustering, retrieval-augmented generation, and agentic reasoning.

## Experiment: Effect of `max_length` and `stride`

In this experiment, we will test the effect of the context window size (`max_length`) and the stride on the number of training samples.

This demonstrates the trade-off between context size, overlap, and dataset size.

In [7]:
settings = [
    (32, 16),
    (64, 32),
    (64, 8),
]

for max_length, stride in settings:
    X_tmp, y_tmp = create_dataset(tokens, max_length, stride)
    print(f"max_length={max_length}, stride={stride} → samples={X_tmp.shape[0]}")

max_length=32, stride=16 → samples=320
max_length=64, stride=32 → samples=159
max_length=64, stride=8 → samples=636


### Experiment Analysis

As `max_length` is increased, there are fewer total samples because each training sample now takes up more tokens.

As `stride` is decreased, there is more overlap between the windows, and this results in more samples being generated. This overlap is very helpful because it allows the model to see more training examples and reinforces learning in different contexts.

This is a trade-off that is inherent in preparing datasets for training Large Language Models.